In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from keras.applications import InceptionV3
from keras.layers import GlobalAveragePooling2D, Dense, Dropout
from keras.models import Sequential
from keras.utils import to_categorical

# Veri yolu
DATADIR = "/content/drive/MyDrive/RetinalImaging/Eyes/"
CATEGORIES = ["Cataract", "Diabetic Retinopathy", "Glaucoma", "Normal"]
IMG_SIZE = 225

# Tüm veriyi depolamak için liste
all_data = []

# Tüm veriyi alma fonksiyonu
def create_all_data():
    for category in CATEGORIES:
        path = os.path.join(DATADIR, category)
        class_num = CATEGORIES.index(category)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img))
                new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                all_data.append([new_array, class_num])
            except Exception as e:
                pass

# Tüm veriyi oluşturma
create_all_data()

# Giriş ve çıkış verilerini ayırma
X = []
y = []
for features, label in all_data:
    X.append(features)
    y.append(label)

In [4]:

from keras.callbacks import ReduceLROnPlateau
from keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D


# Veriyi eğitim ve test setlerine ayırma
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Veriyi uygun şekle dönüştürme
x_train = np.array(x_train)
x_test = np.array(x_test)
y_train = to_categorical(y_train, num_classes=len(CATEGORIES))
y_test = to_categorical(y_test, num_classes=len(CATEGORIES))

# İnceptionV3 modeli
inception_model = InceptionV3(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False)

# Global Average Pooling Layer ekleme
model = Sequential()
model.add(inception_model)
# model.add(Conv2D(12, (3, 3), activation='relu', input_shape=(75, 75, 3)))
# model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Conv2D(24, (3, 3), activation='relu'))
# model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Conv2D(36, (3, 3), activation='relu'))
# model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(GlobalAveragePooling2D())

# Tam bağlantı katmanı ekleme
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.5))
# model.add(Dense(512, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(256, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(128, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(64, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(32, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(16, activation='relu'))
# model.add(Dropout(0.5))
# model.add(Dense(8, activation='relu'))
model.add(Dense(len(CATEGORIES), activation='softmax'))

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=0.00001, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', min_delta=0, patience=5, verbose=1, mode='auto')

# Modeli derleme
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])




87910968/87910968 [==============================] - 0s 0us/step


In [5]:
model.fit(x_train, y_train, epochs=30, batch_size=32, validation_data=(x_test, y_test), callbacks=[reduce_lr, early_stop] )

# Modelin performansını değerlendirme
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

accuracy = accuracy_score(y_true, y_pred_classes)
print("Model Doğruluğu:", accuracy)

print("Sınıflandırma Raporu:")
print(classification_report(y_true, y_pred_classes, target_names=CATEGORIES))

Epoch 1/30
106/106 [==============================] - 83s 327ms/step - loss: 0.6598 - accuracy: 0.7602 - val_loss: 9.9866 - val_accuracy: 0.4396 - lr: 0.0010
Epoch 2/30
106/106 [==============================] - 26s 250ms/step - loss: 0.3725 - accuracy: 0.8630 - val_loss: 8.3303 - val_accuracy: 0.5154 - lr: 0.0010
Epoch 3/30
106/106 [==============================] - 26s 247ms/step - loss: 0.2866 - accuracy: 0.9007 - val_loss: 0.3218 - val_accuracy: 0.9159 - lr: 0.0010
Epoch 4/30
106/106 [==============================] - 26s 250ms/step - loss: 0.2160 - accuracy: 0.9229 - val_loss: 0.3463 - val_accuracy: 0.8697 - lr: 0.0010
Epoch 5/30
106/106 [==============================] - 26s 250ms/step - loss: 0.1768 - accuracy: 0.9416 - val_loss: 0.4144 - val_accuracy: 0.8839 - lr: 0.0010
Epoch 6/30
106/106 [==============================] - 26s 245ms/step - loss: 0.2117 - accuracy: 0.9327 - val_loss: 0.3044 - val_accuracy: 0.9100 - lr: 0.0010
Epoch 7/30
106/106 [==============================] 